In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/cleaned/supply_chain_cleaned.csv')

print("Shape:", df.shape)
print("Columns:", list(df.columns))

Shape: (180519, 22)
Columns: ['order_id', 'order_date', 'ship_date', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'late_delivery_risk', 'delivery_status', 'customer_id', 'customer_segment', 'market', 'order_region', 'order_country', 'order_city', 'category_name', 'product_name', 'sales', 'order_profit_per_order', 'order_item_discount', 'order_item_quantity', 'shipping_mode', 'order_year', 'order_month']


In [3]:
# Shipping delay
df['shipping_delay_days'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']

# Delay severity label
def classify_delay(delay):
    if delay <= 0:
        return 'On Time or Early'
    elif delay <= 2:
        return 'Minor Delay'
    elif delay <= 5:
        return 'Moderate Delay'
    else:
        return 'Severe Delay'

df['delay_severity'] = df['shipping_delay_days'].apply(classify_delay)

# Delay weight multiplier
def delay_weight(delay):
    if delay <= 0:
        return 0
    elif delay <= 2:
        return 1
    elif delay <= 5:
        return 2
    else:
        return 3

df['delay_weight'] = df['shipping_delay_days'].apply(delay_weight)

# Weighted revenue at risk
df['weighted_revenue_at_risk'] = df.apply(
    lambda row: row['sales'] * row['delay_weight']
    if row['late_delivery_risk'] == 1 else 0,
    axis=1
)

# Profit margin per order
df['profit_margin_pct'] = (
    df['order_profit_per_order'] / df['sales'].replace(0, np.nan)
) * 100

# Loss order flag
df['is_loss_order'] = (df['order_profit_per_order'] < 0).astype(int)

# Revenue band
def revenue_band(sales):
    if sales < 50:
        return 'Low'
    elif sales < 200:
        return 'Medium'
    else:
        return 'High'

df['revenue_band'] = df['sales'].apply(revenue_band)

print("New columns added:")
print(df[['shipping_delay_days', 'delay_severity', 'delay_weight',
          'weighted_revenue_at_risk', 'profit_margin_pct',
          'is_loss_order', 'revenue_band']].head(10))

New columns added:
   shipping_delay_days    delay_severity  delay_weight  \
0                   -1  On Time or Early             0   
1                    1       Minor Delay             1   
2                    0  On Time or Early             0   
3                   -1  On Time or Early             0   
4                   -2  On Time or Early             0   
5                    2       Minor Delay             1   
6                    1       Minor Delay             1   
7                    1       Minor Delay             1   
8                    1       Minor Delay             1   
9                    1       Minor Delay             1   

   weighted_revenue_at_risk  profit_margin_pct  is_loss_order revenue_band  
0                      0.00          27.841342              0         High  
1                    327.75         -75.999999              1         High  
2                      0.00         -75.600305              1         High  
3                      0.00       

In [5]:
df.to_csv('../data/cleaned/supply_chain_cleaned.csv', index=False)

print("Saved successfully.")
print("Shape:", df.shape)
print("Total columns:", len(df.columns))
print("Loss orders:", df['is_loss_order'].sum())
print("Loss order rate:", round(df['is_loss_order'].mean() * 100, 2), "%")
print("Avg shipping delay:", round(df['shipping_delay_days'].mean(), 2), "days")

Saved successfully.
Shape: (180519, 29)
Total columns: 29
Loss orders: 33784
Loss order rate: 18.71 %
Avg shipping delay: 0.57 days
